In [1]:
import os

In [2]:
os.getcwd()

'd:\\Berlin_House_Price_Prediction\\Berlin_House_Price_Prediction\\research'

In [3]:
os.chdir('../')

In [4]:
%pwd

'd:\\Berlin_House_Price_Prediction\\Berlin_House_Price_Prediction'

In [ ]:
from dataclasses import dataclass
from pathlib import Path

@dataclass(frozen=True)
class DataTransformationConfig:
    root_directory: Path
    data_path: Path

In [ ]:
from Berlin_House_Price_Prediction.constants import *
from Berlin_House_Price_Prediction.utils.common import read_yaml,create_directories
from Berlin_House_Price_Prediction.entity.config_entity import DataIngestionConfig
from Berlin_House_Price_Prediction.entity.config_entity import DataValidationConfig


# this class has a constructor which reads the yaml files
class ConfigurationManger:
    # Reading yaml file and create root directory
    def __init__(self,
                 config_filepath = CONFIG_FILE_PATH,
                 params_filepath = PARAMS_FILE_PATH,
                 schema_filepath = SCHEMA_FILE_PATH
                 ):
        self.config = read_yaml(config_filepath)
        self.params = read_yaml(params_filepath)
        self.schema = read_yaml(schema_filepath)

        create_directories([self.config.artifacts_root])


    # 
    def get_data_ingestion_config(self)->DataIngestionConfig:
        config = self.config.data_ingestion

        create_directories([config.root_directory])

        data_ingestion_config = DataIngestionConfig(
            root_directory=config.root_directory,
            source_url=config.source_URL,
            local_data_file= config.local_data_file,
            unzip_dir= config.unzip_dir
        )
        
        return data_ingestion_config
    

    def get_data_validation_config(self)->DataValidationConfig:
        config = self.config.data_validation
        schema = self.schema.COLUMNS

        create_directories([config.root_directory])

        data_validation_config = DataValidationConfig(
            root_directory= config.root_directory,
            unzip_data_dir= config.unzip_data_dir,
            STATUS_FILE = config.STATUS_FILE,
            all_schema= schema,
        )

        return data_validation_config
    
    def get_data_transformation_config(self)->DataTransformationConfig:
        config = self.config.data_transformation

        create_directories([config.root_directory])

        data_transformation_config = DataTransformationConfig(
            root_directory= config.root_directory,
            data_path= config.data_path,
        )

        return data_transformation_config

In [16]:
from Berlin_House_Price_Prediction.config.configuration import DataValidationConfig
import pandas as pd


class DataValidation:
    def __init__(self, config: DataValidationConfig):
        self.config = config

    def validate_all_columns(self)->bool:
        try:
            validation_status=None
            data = pd.read_csv(self.config.unzip_data_dir)
            all_columns = data.columns
            all_schema_columns = self.config.all_schema.keys()


            for column in all_columns: 
                if column not in all_schema_columns:
                    validation_status = False
                    with open(self.config.STATUS_FILE, 'w') as f:
                        f.write(f"Validation status: {validation_status}")
                else:
                    validation_status = True
                    with open(self.config.STATUS_FILE, 'w') as f:
                        f.write(f"Validation status: {validation_status}")
                if data[column].dtype != self.config.all_schema[column]:
                    with open(self.config.STATUS_FILE, 'w') as f:
                        f.write(f"Data type mismatch for column:{column} \nexpected: {self.config.all_schema[column]}\n received:{data[column].dtype}")
                
                
            return validation_status

        except Exception as e:
            raise e

In [17]:
try:
    config = ConfigurationManger()
    data_validation_config = config.get_data_validation_config()
    data_validation = DataValidation(config=data_validation_config)
    data_validation.validate_all_columns()
except Exception as e:
    raise e
    

[2026-03-03 16:53:03,782: INFO: common: yaml file: <_io.TextIOWrapper name='config\\config.yaml' mode='r' encoding='cp1252'> loaded successfully]
[2026-03-03 16:53:03,784: INFO: common: yaml file: <_io.TextIOWrapper name='params.yaml' mode='r' encoding='cp1252'> loaded successfully]
[2026-03-03 16:53:03,786: INFO: common: yaml file: <_io.TextIOWrapper name='schema.yaml' mode='r' encoding='cp1252'> loaded successfully]
[2026-03-03 16:53:03,788: INFO: common: created directory at: artifacts]
[2026-03-03 16:53:03,790: INFO: common: created directory at: artifacts/data_validation]
